# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaaDasim05/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a **Random Forest classifier** for the Refresh / Content Opportunity Scoring lane.

The model fits the task because the target is a binary proxy for whether a content item experiences a meaningful future decline in search impressions. Random Forest can combine several historical signals and capture non-linear relationships without requiring the relationships to be specified manually.

I will use the model's predicted probability of decline as the opportunity score. Pages will then be ranked by that score so the output can support a limited content-review queue.

The goal is not to reward model complexity. The model must be evaluated against the transparent Week-4 baseline on the same data, metric, and client-level split.

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# Check the months available for the experiment
months = con.sql(f"""
    SELECT
        month,
        COUNT(*) AS rows
    FROM {TABLES['fact_daily']}
    WHERE month IN ('2026-03', '2026-04')
    GROUP BY month
    ORDER BY month
""").df()

months

,month,rows
0,2026-03,9841378
1,2026-04,10424730


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a time-aware feature/outcome design. Features are calculated only from March 2026, and the target is calculated from April 2026, so future outcome information does not enter the feature set.

The primary validation split is a client-level `GroupShuffleSplit` using `client_hash_id`. This keeps all content from a client in either training or testing, rather than allowing pages from the same client to appear in both.

The final evaluation will compare the Random Forest with the Week-4 baseline on the same held-out clients and using the same ranking metric, Precision@50.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
# March = information available at decision time
# April = future outcome

march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,

        AVG(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS avg_position_march,

        SUM(sessions_organic) AS organic_sessions_march,

        COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_impressions > 0
        ) AS days_with_impressions_march,

        -- Past-only trend signal for the baseline:
        SUM(
            CASE
                WHEN report_date < DATE '2026-03-16'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS impressions_first15,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-16'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS impressions_last16

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY 1, 2
""").df()

# Future outcome: April impressions
april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-04-01'
      AND report_date < DATE '2026-05-01'

    GROUP BY 1, 2
""").df()

data = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Future decline proxy
data["is_declining"] = (
    data["impressions_april"]
    < 0.8 * data["impressions_march"]
).astype(int)

print(f"Rows with March features + April outcome: {len(data):,}")
print(f"Clients: {data['client_hash_id'].nunique():,}")
print(f"Decline rate: {data['is_declining'].mean():.3f}")

data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with March features + April outcome: 331,436
Clients: 55
Decline rate: 0.284


,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,organic_sessions_march,days_with_impressions_march,impressions_first15,impressions_last16,impressions_april,is_declining
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,31,4173.0,2350.0,6787.0,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,31,245.0,208.0,405.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,31,3705.0,1925.0,8475.0,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,1.0,31,2440.0,2504.0,6091.0,0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,0.0,21,14.0,28.0,67.0,0


In [4]:
content_schema = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{REL}/dim_content.parquet'
    )
    LIMIT 0
""").df()

print(content_schema.columns.tolist())

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# ---------------------------------------------------------
# Add decision-time content freshness from dim_content
# ---------------------------------------------------------

content_dates = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        content_created_date,
        content_type,
        word_count
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()

content_dates["content_updated_date"] = pd.to_datetime(
    content_dates["content_updated_date"],
    errors="coerce"
)

# Decision date = end of March 2026
decision_date = pd.Timestamp("2026-03-31")

content_dates["days_since_update"] = (
    decision_date - content_dates["content_updated_date"]
).dt.days

# Avoid negative values from future-dated timestamps
content_dates["days_since_update"] = (
    content_dates["days_since_update"]
    .clip(lower=0)
)

# Keep only fields needed here
content_dates = content_dates[
    [
        "client_hash_id",
        "content_hash_id",
        "days_since_update",
        "content_type",
        "word_count"
    ]
]

data = data.merge(
    content_dates,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print(f"Rows after content join: {len(data):,}")
print(f"Missing update dates: {data['days_since_update'].isna().sum():,}")

data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after content join: 331,436
Missing update dates: 0


,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,organic_sessions_march,days_with_impressions_march,impressions_first15,impressions_last16,impressions_april,is_declining,days_since_update,content_type,word_count
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,31,4173.0,2350.0,6787.0,0,0,keyword article,2123
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,31,245.0,208.0,405.0,0,0,keyword article,<NA>
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,31,3705.0,1925.0,8475.0,0,0,keyword article,2546
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,1.0,31,2440.0,2504.0,6091.0,0,0,keyword article,2330
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,0.0,21,14.0,28.0,67.0,0,0,keyword article,<NA>


In [6]:
# ---------------------------------------------------------
# Week-4-style transparent baseline
# ---------------------------------------------------------

baseline_df = data.copy()

# Historical March trend, knowable at March 31
baseline_df["march_trend_pct"] = np.where(
    baseline_df["impressions_first15"] > 0,
    (
        baseline_df["impressions_last16"]
        - baseline_df["impressions_first15"]
    )
    / baseline_df["impressions_first15"] * 100,
    0
)

# 60% negative trend + 40% staleness
trend_component = (
    (-baseline_df["march_trend_pct"])
    .clip(lower=0, upper=100) / 100
)

stale_component = (
    baseline_df["days_since_update"]
    .clip(lower=0, upper=365) / 365
)

baseline_df["baseline_score"] = 100 * (
    0.60 * trend_component +
    0.40 * stale_component
)

baseline_df = baseline_df.dropna(
    subset=["baseline_score", "is_declining"]
).copy()

print(f"Rows usable for baseline/model comparison: {len(baseline_df):,}")
print(f"Clients: {baseline_df['client_hash_id'].nunique():,}")

Rows usable for baseline/model comparison: 331,436
Clients: 55


In [7]:
from sklearn.model_selection import GroupShuffleSplit

groups = baseline_df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(baseline_df, baseline_df["is_declining"], groups=groups)
)

train_data = baseline_df.iloc[train_idx].copy()
test_data = baseline_df.iloc[test_idx].copy()

print("Training clients:", train_data["client_hash_id"].nunique())
print("Testing clients:", test_data["client_hash_id"].nunique())
print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

Training clients: 41
Testing clients: 14
Training rows: 297081
Testing rows: 34355


In [8]:
def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)[:k]

    return labels[order].mean()

In [9]:
from sklearn.ensemble import RandomForestClassifier

model_features = [
    "impressions_march",
    "clicks_march",
    "avg_position_march",
    "organic_sessions_march",
    "days_with_impressions_march",
    "days_since_update",
    "word_count"
]

model_train = train_data.dropna(
    subset=model_features + ["is_declining"]
).copy()

model_test = test_data.dropna(
    subset=model_features + ["is_declining"]
).copy()

X_train = model_train[model_features]
y_train = model_train["is_declining"]

X_test = model_test[model_features]
y_test = model_test["is_declining"]

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

model_scores = rf.predict_proba(X_test)[:, 1]

print("Model Precision@50:",
      round(precision_at_k(model_scores, y_test, 50), 3))

Model Precision@50: 0.52


In [10]:
# Use exactly the same model-test rows for both methods

baseline_scores = model_test["baseline_score"].values
labels = model_test["is_declining"].values

baseline_precision = precision_at_k(
    baseline_scores,
    labels,
    50
)

model_precision = precision_at_k(
    model_scores,
    labels,
    50
)

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_precision,
        model_precision
    ]
})

comparison

,method,Precision@50
0,Week-4 baseline,0.46
1,Random Forest,0.52


### Model vs baseline result

The Random Forest achieved **0.52 Precision@50**, compared with **0.46** for the Week-4 transparent baseline, on the same 14 held-out clients.

This is an absolute improvement of **0.06 Precision@50**. In practical terms, the model placed more future-declining content items among its top 50 recommendations than the baseline in this particular client-level test split.

This is evidence that combining multiple historical features can improve the prioritization ranking over the simple hand-written rule. It is still a directional result from one grouped split, not proof that the model will outperform the baseline for every future client.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
importance = (
    pd.Series(
        rf.feature_importances_,
        index=model_features
    )
    .sort_values(ascending=False)
)

print("Random Forest feature importance:")
print(importance)

Random Forest feature importance:
word_count                     0.359630
avg_position_march             0.237110
impressions_march              0.195138
days_with_impressions_march    0.081174
clicks_march                   0.055414
organic_sessions_march         0.053654
days_since_update              0.017880
dtype: float64


In [12]:
test_analysis = model_test[
    [
        "client_hash_id",
        "content_hash_id",
        "is_declining",
        "baseline_score"
    ]
].copy()

test_analysis["model_score"] = model_scores

# Actual label from the future window
test_analysis["prediction"] = (
    test_analysis["model_score"] >= 0.5
).astype(int)

false_positives = test_analysis[
    (test_analysis["prediction"] == 1) &
    (test_analysis["is_declining"] == 0)
]

false_negatives = test_analysis[
    (test_analysis["prediction"] == 0) &
    (test_analysis["is_declining"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head(10))

print("\nExample false negatives:")
display(false_negatives.head(10))

False positives: 3814
False negatives: 2662

Example false positives:


,client_hash_id,content_hash_id,is_declining,baseline_score,model_score,prediction
9861,client_c182d11e4862a37d,content_700ed17f1ed3d4cf,0,0.000000,0.993333,1
9871,client_c182d11e4862a37d,content_8bb670a457359936,0,0.000000,0.766667,1
9882,client_c182d11e4862a37d,content_c7034ce2e244eb59,0,36.302521,0.873333,1
9891,client_c182d11e4862a37d,content_03490659f49a9a84,0,0.000000,0.713333,1
9910,client_c182d11e4862a37d,content_205fd738fcb9a050,0,54.000000,0.603333,1
9925,client_c182d11e4862a37d,content_6bf90406b4d4a8e2,0,0.000000,0.643333,1
9931,client_c182d11e4862a37d,content_5fc01867856d36c7,0,0.000000,0.840000,1
11837,client_59256b0571e0c970,content_717f3b70ec58811f,0,0.000000,0.780000,1
11840,client_59256b0571e0c970,content_12add63b3ca36b9b,0,35.490733,0.626667,1
14832,client_e547b89c05043229,content_e67934818ca184a1,0,0.000000,0.600000,1



Example false negatives:


,client_hash_id,content_hash_id,is_declining,baseline_score,model_score,prediction
9866,client_c182d11e4862a37d,content_07391c5144deb0b6,1,24.742268,0.496667,0
9876,client_c182d11e4862a37d,content_90ee2c180a7f9bb7,1,30.576606,0.496667,0
14833,client_e547b89c05043229,content_9634c35544bcc47b,1,0.000000,0.493333,0
14841,client_e547b89c05043229,content_c3337ad311c348d7,1,0.000000,0.436667,0
14844,client_e547b89c05043229,content_1f05a108a2caf0b5,1,0.000000,0.390000,0
14845,client_e547b89c05043229,content_bf328a70f653d97a,1,0.000000,0.393333,0
14852,client_e547b89c05043229,content_ace2fd56a9c476c3,1,1.442308,0.440000,0
14864,client_e547b89c05043229,content_533176ee7084955d,1,0.000000,0.410000,0
14872,client_e547b89c05043229,content_a344e29b03064479,1,0.000000,0.480000,0
14877,client_e547b89c05043229,content_2f1604b73f465db2,1,0.000000,0.423333,0


## 4. Errors and interpretation

The Random Forest's feature importance was highest for `word_count` (0.360), followed by `avg_position_march` (0.237) and `impressions_march` (0.195). The remaining features contributed less individually, and `days_since_update` had relatively low importance (0.018).

This differs from my Week-4 baseline, which relied heavily on staleness and negative historical trend. In this model, content size, historical search position, and historical impressions carried more of the learned signal. These importances describe how the fitted Random Forest used the available features in this particular experiment; they do not establish that these features cause future performance declines.

The model produced 3,814 false positives and 2,662 false negatives on the held-out test rows. False positives indicate pages that the model ranked as likely to decline but did not meet the future decline proxy. False negatives indicate pages that did decline but received lower predicted probabilities.

Several false-negative examples have predicted probabilities close to the 0.50 classification threshold, suggesting that some errors occur around borderline cases rather than only on obviously misclassified pages.

For the refresh-ranking use case, the main result remains the Precision@50 comparison rather than the binary 0.50 threshold. The model achieved 0.52 Precision@50 compared with 0.46 for the Week-4 baseline on the same held-out clients.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.